<a href="https://colab.research.google.com/github/sergioGarcia91/SeismicUP/blob/main/04_Retornos_y_fractales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

En este notebook se procederá a calcular los períodos de retorno y la dimensión fractal de los catálogos. Para ello, se utilizarán los resultados de los dataframes generados en el Notebook 03, los cuales contienen la información correspondiente a los parámetros a-value, b-value y Mc.

# Inicio

In [ ]:
!git clone https://github.com/sergioGarcia91/SeismicUP.git

In [ ]:
# para incluir la libreria clonada
import sys
sys.path.append("/content/SeismicUP")

import seismicup as sup

In [ ]:
!pip -q install python-calamine

In [ ]:
!pip3 install contextily

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import seaborn as sns
import os
import re
import contextily as cx #para el basemap en geopandas
import xyzservices.providers as xyz #para escoger el basemap
import seismicup as sup
import urllib.request
import matplotlib.font_manager as fm
import random

from matplotlib.colors import LogNorm # para la escala logaritmica de los colores
from matplotlib.ticker import LogFormatter, LogLocator # escala log
from scipy import stats # regresion lineal

# Conectar al Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cambiar Fuente

In [ ]:
sup.plots.get_TimesNewRoman_font()

# Paths

In [ ]:
path_save_figures = '/content/drive/MyDrive/Contratos/20250801_UIS_CO2/Notebooks/Figuras_04'

In [ ]:
path_df_N03 = '/content/drive/MyDrive/Contratos/20250801_UIS_CO2/Notebooks/Figuras_03'

In [ ]:
path_datasets = '/content/SeismicUP/Datasets_'

os.listdir(path_datasets)

# Cargar catalogos

In [ ]:
lista_catalogos = ['Cat_sis_SGC_jun1993_feb2018',
                   'Cat_sis_SGC_mar2018_2024',
                   'Cat_sis_SGC_TECTO_feb2014_2024',
                   'Cat_sis_SGC_LBG_jun1993_2021']

for i in lista_catalogos:
  cantidad_archivos = len(os.listdir(os.path.join(path_datasets, i)))
  print(f'{i} = ', cantidad_archivos)

In [ ]:
df_general = pd.DataFrame(columns=['Fecha-Hora UTC',
                                   'Latitud', # grados
                                   'Longitud', # grados
                                   'Profundidad [km]',
                                   'Magnitud',
                                   'Tipo Magnitud',
                                   'Error Latitud [km]',
                                   'Error Longitud [km]',
                                   'Error Profundidad [km]',
                                   'Numero de Fases',
                                   'RMS [seg]',
                                   'Gap', # grados
                                   'Departamento',
                                   'Municipio',
                                   ])

df_general.head()

## Catalogo SGC 1

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_jun1993_feb2018')
os.listdir(path_catalogo)

df_SGC_1 = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_1(path_catalogo=path_catalogo,
                                            file_excel=i)

    df_SGC_1 = pd.concat([df_SGC_1, df_temp_1], ignore_index=True)
    df_SGC_1.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_1['Catalogo'] = 'SGC 1'
df_SGC_1['Gap'] = df_SGC_1['Gap'].astype(float)
df_SGC_1.info()

In [ ]:
df_SGC_1.head()

## Catalogo SGC 2

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_mar2018_2024')
os.listdir(path_catalogo)

df_SGC_2 = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_2(path_catalogo=path_catalogo,
                                            file_excel=i)

    df_SGC_2 = pd.concat([df_SGC_2, df_temp_1], ignore_index=True)
    df_SGC_2.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_2['Catalogo'] = 'SGC 2'
df_SGC_2['Gap'] = df_SGC_2['Gap'].astype(float)
df_SGC_2.info()

In [ ]:
df_SGC_2.head()

## Catalogo TECTO

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_TECTO_feb2014_2024')
os.listdir(path_catalogo)

df_SGC_TECTO = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx') and (i != 'excel_estaciones.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_TECTO(path_catalogo=path_catalogo,
                                                file_excel=i)

    print(df_temp_1.shape)
    df_SGC_TECTO = pd.concat([df_SGC_TECTO, df_temp_1], ignore_index=True)
    df_SGC_TECTO.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_TECTO['Catalogo'] = 'TECTO'
df_SGC_TECTO['Profundidad [km]'] = df_SGC_TECTO['Profundidad [km]'].astype(float)
df_SGC_TECTO['Gap'] = df_SGC_TECTO['Gap'].astype(float)
df_SGC_TECTO.info()

In [ ]:
df_SGC_TECTO.head()

## Catalogo LBG

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_LBG_jun1993_2021')
os.listdir(path_catalogo)

df_SGC_LBG = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_LBG(path_catalogo=path_catalogo,
                                              file_excel=i)

    print(df_temp_1.shape)
    df_SGC_LBG = pd.concat([df_SGC_LBG, df_temp_1], ignore_index=True)
    df_SGC_LBG.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_LBG['Catalogo'] = 'LBG'
df_SGC_LBG.info()

In [ ]:
df_SGC_LBG.head()

# Unir catalogos

In [ ]:
df_unido = pd.concat([df_SGC_1, df_SGC_2, df_SGC_TECTO, df_SGC_LBG], ignore_index=True)
df_unido.reset_index(drop=True, inplace=True)

df_unido['Tiempo'] = pd.to_datetime(df_unido['Fecha-Hora UTC'])

df_unido.head()

In [ ]:
df_unido['Catalogo'].unique()

In [ ]:
df_unido.info()

In [ ]:
df_unido.describe().round(2)

## Duplicados

In [ ]:
df_unido.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# 22154 duplicados

In [ ]:
df_unido['Tipo Magnitud'].unique()

In [ ]:
df_unido['Tipo Magnitud 2'] = df_unido['Tipo Magnitud'].str.lower()
df_unido['Tipo Magnitud 2'].unique()

In [ ]:
df_unido.describe()

In [ ]:
df_unido.sort_values(by=['Tiempo', 'Tipo Magnitud 2'],
                     ascending=True,
                     inplace=True)

Se considerarán únicamente los eventos que cuenten con ML como tipo de magnitud.

In [ ]:
filtro_ml = df_unido['Tipo Magnitud 2'].str.contains('ml')

df_unido[filtro_ml]['Tipo Magnitud 2'].unique()

In [ ]:
df_unido.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# 22154 duplicados

In [ ]:
df_filtrado = df_unido.copy()
df_filtrado.drop_duplicates(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first', inplace=True)
df_filtrado.reset_index(drop=True, inplace=True)

In [ ]:
df_filtrado.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# Ya no se tienen duplicados

In [ ]:
df_filtrado.describe().round(2)

## Geodataframe

In [ ]:
gdf_filtrado = gpd.GeoDataFrame(
    df_filtrado, geometry=gpd.points_from_xy(df_filtrado.Longitud, df_filtrado.Latitud), crs="EPSG:4326"
)

In [ ]:
gdf_filtrado = gdf_filtrado.to_crs(9377)

In [ ]:
df_filtrado['X [m]'] = gdf_filtrado.get_coordinates()['x']
df_filtrado['Y [m]'] = gdf_filtrado.get_coordinates()['y']

df_filtrado['X [km]'] = df_filtrado['X [m]'] / 1000
df_filtrado['Y [km]'] = df_filtrado['Y [m]'] / 1000

df_filtrado.head()

## Punto referencia Campo Colorado



In [ ]:
punto_xy_CampoColorado = (4919222.973, 2308553.476) # x, y en metros

# se va calcular la distancia de los eventos al centroide o punto de referencia
# del Campo Colorado solo en XY mas no en profundidad

df_filtrado['Distancia [m]'] = np.sqrt((df_filtrado['X [m]'] - punto_xy_CampoColorado[0])**2 + (df_filtrado['Y [m]'] - punto_xy_CampoColorado[1])**2)
df_filtrado['Distancia [km]'] = df_filtrado['Distancia [m]'] / 1000

df_filtrado.describe().round(2)

## Subcatalogos

In [ ]:
limites_BSN = {'longitud': [-73.40, -72.80],
               'latitud': [6.50, 7.10]}
limites_CC = {'longitud': [-74.19, -73.26],
              'latitud': [6.32, 7.25]}


In [ ]:
df_FC = df_filtrado.copy()

filtro_BSN = (df_filtrado['Longitud'] >= limites_BSN['longitud'][0]) & (df_filtrado['Longitud'] <= limites_BSN['longitud'][1]) & (df_filtrado['Latitud'] >= limites_BSN['latitud'][0]) & (df_filtrado['Latitud'] <= limites_BSN['latitud'][1])
df_BSN = df_filtrado[filtro_BSN].copy()

filtro_CC = (df_filtrado['Longitud'] >= limites_CC['longitud'][0]) & (df_filtrado['Longitud'] <= limites_CC['longitud'][1]) & (df_filtrado['Latitud'] >= limites_CC['latitud'][0]) & (df_filtrado['Latitud'] <= limites_CC['latitud'][1])
df_CC = df_filtrado[filtro_CC].copy()


In [ ]:
df_FC.describe().round(2)

In [ ]:
df_BSN.describe().round(2)

In [ ]:
df_CC.describe().round(2)

# DF Notebook 03

In [ ]:
lista_df = os.listdir(path_df_N03)
lista_df = [i for i in lista_df if 'csv' in i]

lista_df

El archivo que se utilizará en este análisis es `df_GR_todos.csv`, el cual contiene los valores estimados para todos los catálogos, considerando todo el rango temporal. Adicionalmente, el archivo incluye los resultados correspondientes a los subcatálogos somero y profundo definidos a partir de una profundidad de corte de 60 km (Notebook 03). En consecuencia, los cálculos se realizarán únicamente para estos subcatálogos y no para los valores acumulados en el tiempo


In [ ]:
df_GR_todos = pd.read_csv(os.path.join(path_df_N03, 'df_GR_todos.csv'),
                          decimal=',',
                          sep= ';',
                          index_col=0)

df_GR_todos.info()

In [ ]:
df_GR_todos.round(3)

In [ ]:
df_GR_todos.loc['b_mean', 'FC']

## Retorno todos

In [ ]:
magnitud_minima_retorno = 0.1
magnitud_maxima_retorno = 7
steps_magnitudes = 0.2
print_texto = False

### FC todos

In [ ]:
df_FC.describe().round(2)

In [ ]:
total_time = df_FC['Tiempo'].max() - df_FC['Tiempo'].min()
total_time = total_time / np.timedelta64(1, 'D')
total_time = total_time / 365.25

total_time # en años

In [ ]:
dict_retornos_FC_todos = sup.stats.periodo_retorno(a_value = df_GR_todos.loc['a_mean', 'FC'],
                                                   b_value = df_GR_todos.loc['b_mean', 'FC'],
                                                   magnitud_minima = magnitud_minima_retorno,
                                                   magnitud_maxima = magnitud_maxima_retorno,
                                                   steps_ = steps_magnitudes,
                                                   periodo_years = total_time,
                                                   retorno_years = [1, 10, 50, 100],
                                                   texto = print_texto)

dict_retornos_FC_todos.keys()

In [ ]:
dict_retornos_FC_todos.keys()

In [ ]:
dict_retornos_FC_todos['m']

In [ ]:
dict_retornos_FC_todos['N']

In [ ]:
dict_retornos_FC_todos['P_1year']

In [ ]:
dict_retornos_FC_todos['P_all_time']

In [ ]:
dict_retornos_FC_todos['P_retornos']

In [ ]:
dict_retornos_FC_todos['P_retornos'][0] # es una lista

In [ ]:
dict_retornos_FC_todos['theta_years']

In [ ]:
np.array(dict_retornos_FC_todos['theta_days']).astype(float).round(5)

In [ ]:
3.2000000e-03 * 24 * 60

In [ ]:
dict_retornos_FC_todos['theta_months']

In [ ]:
dict_retornos_FC_todos.keys()

In [ ]:
np.array(dict_retornos_FC_todos['m']).shape

In [ ]:
np.array(dict_retornos_FC_todos['theta_years']).shape

In [ ]:
dict_retornos_FC_todos['P_retornos'][0]

In [ ]:
df_retornos_todos = {'Magnitud': [],
                     'Periodo retorno': [],
                     '1 año': [],
                     '10 año': [],
                     '50 año': [],
                     '100 año': []}

for mag, retorno in zip(dict_retornos_FC_todos['m'],
                        dict_retornos_FC_todos['theta_years']):
  df_retornos_todos['Magnitud'].append(mag)
  df_retornos_todos['Periodo retorno'].append(retorno)

for item in dict_retornos_FC_todos['P_retornos']:
  df_retornos_todos['1 año'].append(item['value'][0])
  df_retornos_todos['10 año'].append(item['value'][1])
  df_retornos_todos['50 año'].append(item['value'][2])
  df_retornos_todos['100 año'].append(item['value'][3])


df_retornos_todos = pd.DataFrame(df_retornos_todos)

df_retornos_todos.to_csv(os.path.join(path_save_figures, 'df_retornos_FC_todos.csv'),
                         decimal=',',
                         sep=';',
                         index=True)

df_retornos_todos

### BSN todos

In [ ]:
df_BSN.describe().round(2)

In [ ]:
total_time = df_BSN['Tiempo'].max() - df_BSN['Tiempo'].min()
total_time = total_time / np.timedelta64(1, 'D')
total_time = total_time / 365.25

total_time # en años

In [ ]:
dict_retornos_BSN_todos = sup.stats.periodo_retorno(a_value = df_GR_todos.loc['a_mean', 'BSN'],
                                                   b_value = df_GR_todos.loc['b_mean', 'BSN'],
                                                   magnitud_minima = magnitud_minima_retorno,
                                                   magnitud_maxima = magnitud_maxima_retorno,
                                                   steps_ = steps_magnitudes,
                                                   periodo_years = total_time,
                                                   retorno_years = [1, 10, 50, 100],
                                                   texto = print_texto)

dict_retornos_BSN_todos.keys()

In [ ]:
dict_retornos_BSN_todos.keys()

In [ ]:
df_retornos_todos = {'Magnitud': [],
                     'Periodo retorno': [],
                     '1 año': [],
                     '10 año': [],
                     '50 año': [],
                     '100 año': []}

for mag, retorno in zip(dict_retornos_BSN_todos['m'],
                        dict_retornos_BSN_todos['theta_years']):
  df_retornos_todos['Magnitud'].append(mag)
  df_retornos_todos['Periodo retorno'].append(retorno)

for item in dict_retornos_BSN_todos['P_retornos']:
  df_retornos_todos['1 año'].append(item['value'][0])
  df_retornos_todos['10 año'].append(item['value'][1])
  df_retornos_todos['50 año'].append(item['value'][2])
  df_retornos_todos['100 año'].append(item['value'][3])


df_retornos_todos = pd.DataFrame(df_retornos_todos)

df_retornos_todos.to_csv(os.path.join(path_save_figures, 'df_retornos_BSN_todos.csv'),
                         decimal=',',
                         sep=';',
                         index=True)

df_retornos_todos

### CC todos

In [ ]:
df_CC.describe().round(2)

In [ ]:
total_time = df_CC['Tiempo'].max() - df_CC['Tiempo'].min()
total_time = total_time / np.timedelta64(1, 'D')
total_time = total_time / 365.25

total_time # en años

In [ ]:
dict_retornos_CC_todos = sup.stats.periodo_retorno(a_value = df_GR_todos.loc['a_mean', 'CC'],
                                                   b_value = df_GR_todos.loc['b_mean', 'CC'],
                                                   magnitud_minima = magnitud_minima_retorno,
                                                   magnitud_maxima = magnitud_maxima_retorno,
                                                   steps_ = steps_magnitudes,
                                                   periodo_years = total_time,
                                                   retorno_years = [1, 10, 50, 100],
                                                   texto = print_texto)

dict_retornos_CC_todos.keys()

In [ ]:
dict_retornos_CC_todos.keys()

In [ ]:
df_retornos_todos = {'Magnitud': [],
                     'Periodo retorno': [],
                     '1 año': [],
                     '10 año': [],
                     '50 año': [],
                     '100 año': []}

for mag, retorno in zip(dict_retornos_CC_todos['m'],
                        dict_retornos_CC_todos['theta_years']):
  df_retornos_todos['Magnitud'].append(mag)
  df_retornos_todos['Periodo retorno'].append(retorno)

for item in dict_retornos_CC_todos['P_retornos']:
  df_retornos_todos['1 año'].append(item['value'][0])
  df_retornos_todos['10 año'].append(item['value'][1])
  df_retornos_todos['50 año'].append(item['value'][2])
  df_retornos_todos['100 año'].append(item['value'][3])


df_retornos_todos = pd.DataFrame(df_retornos_todos)

df_retornos_todos.to_csv(os.path.join(path_save_figures, 'df_retornos_CC_todos.csv'),
                         decimal=',',
                         sep=';',
                         index=True)

df_retornos_todos

## Retorno someros

In [ ]:
magnitud_minima_retorno = 0.1
magnitud_maxima_retorno = 7
steps_magnitudes = 0.2
print_texto = False

### FC todos

In [ ]:
df_FC.describe().round(2)

In [ ]:
total_time = df_FC['Tiempo'].max() - df_FC['Tiempo'].min()
total_time = total_time / np.timedelta64(1, 'D')
total_time = total_time / 365.25

total_time # en años

In [ ]:
dict_retornos_FC_todos = sup.stats.periodo_retorno(a_value = df_GR_todos.loc['a_mean', 'FC somero'],
                                                   b_value = df_GR_todos.loc['b_mean', 'FC somero'],
                                                   magnitud_minima = magnitud_minima_retorno,
                                                   magnitud_maxima = magnitud_maxima_retorno,
                                                   steps_ = steps_magnitudes,
                                                   periodo_years = total_time,
                                                   retorno_years = [1, 10, 50, 100],
                                                   texto = print_texto)

dict_retornos_FC_todos.keys()

In [ ]:
dict_retornos_FC_todos.keys()

In [ ]:
df_retornos_todos = {'Magnitud': [],
                     'Periodo retorno': [],
                     '1 año': [],
                     '10 año': [],
                     '50 año': [],
                     '100 año': []}

for mag, retorno in zip(dict_retornos_FC_todos['m'],
                        dict_retornos_FC_todos['theta_years']):
  df_retornos_todos['Magnitud'].append(mag)
  df_retornos_todos['Periodo retorno'].append(retorno)

for item in dict_retornos_FC_todos['P_retornos']:
  df_retornos_todos['1 año'].append(item['value'][0])
  df_retornos_todos['10 año'].append(item['value'][1])
  df_retornos_todos['50 año'].append(item['value'][2])
  df_retornos_todos['100 año'].append(item['value'][3])


df_retornos_todos = pd.DataFrame(df_retornos_todos)

df_retornos_todos.to_csv(os.path.join(path_save_figures, 'df_retornos_FC_somero.csv'),
                         decimal=',',
                         sep=';',
                         index=True)

df_retornos_todos

### BSN todos

In [ ]:
df_BSN.describe().round(2)

In [ ]:
total_time = df_BSN['Tiempo'].max() - df_BSN['Tiempo'].min()
total_time = total_time / np.timedelta64(1, 'D')
total_time = total_time / 365.25

total_time # en años

In [ ]:
dict_retornos_BSN_todos = sup.stats.periodo_retorno(a_value = df_GR_todos.loc['a_mean', 'BSN somero'],
                                                   b_value = df_GR_todos.loc['b_mean', 'BSN somero'],
                                                   magnitud_minima = magnitud_minima_retorno,
                                                   magnitud_maxima = magnitud_maxima_retorno,
                                                   steps_ = steps_magnitudes,
                                                   periodo_years = total_time,
                                                   retorno_years = [1, 10, 50, 100],
                                                   texto = print_texto)

dict_retornos_BSN_todos.keys()

In [ ]:
dict_retornos_BSN_todos.keys()

In [ ]:
df_retornos_todos = {'Magnitud': [],
                     'Periodo retorno': [],
                     '1 año': [],
                     '10 año': [],
                     '50 año': [],
                     '100 año': []}

for mag, retorno in zip(dict_retornos_BSN_todos['m'],
                        dict_retornos_BSN_todos['theta_years']):
  df_retornos_todos['Magnitud'].append(mag)
  df_retornos_todos['Periodo retorno'].append(retorno)

for item in dict_retornos_BSN_todos['P_retornos']:
  df_retornos_todos['1 año'].append(item['value'][0])
  df_retornos_todos['10 año'].append(item['value'][1])
  df_retornos_todos['50 año'].append(item['value'][2])
  df_retornos_todos['100 año'].append(item['value'][3])


df_retornos_todos = pd.DataFrame(df_retornos_todos)

df_retornos_todos.to_csv(os.path.join(path_save_figures, 'df_retornos_BSN_somero.csv'),
                         decimal=',',
                         sep=';',
                         index=True)

df_retornos_todos

### CC todos

In [ ]:
df_CC.describe().round(2)

In [ ]:
total_time = df_CC['Tiempo'].max() - df_CC['Tiempo'].min()
total_time = total_time / np.timedelta64(1, 'D')
total_time = total_time / 365.25

total_time # en años

In [ ]:
dict_retornos_CC_todos = sup.stats.periodo_retorno(a_value = df_GR_todos.loc['a_mean', 'CC somero'],
                                                   b_value = df_GR_todos.loc['b_mean', 'CC somero'],
                                                   magnitud_minima = magnitud_minima_retorno,
                                                   magnitud_maxima = magnitud_maxima_retorno,
                                                   steps_ = steps_magnitudes,
                                                   periodo_years = total_time,
                                                   retorno_years = [1, 10, 50, 100],
                                                   texto = print_texto)

dict_retornos_CC_todos.keys()

In [ ]:
dict_retornos_CC_todos.keys()

In [ ]:
df_retornos_todos = {'Magnitud': [],
                     'Periodo retorno': [],
                     '1 año': [],
                     '10 año': [],
                     '50 año': [],
                     '100 año': []}

for mag, retorno in zip(dict_retornos_CC_todos['m'],
                        dict_retornos_CC_todos['theta_years']):
  df_retornos_todos['Magnitud'].append(mag)
  df_retornos_todos['Periodo retorno'].append(retorno)

for item in dict_retornos_CC_todos['P_retornos']:
  df_retornos_todos['1 año'].append(item['value'][0])
  df_retornos_todos['10 año'].append(item['value'][1])
  df_retornos_todos['50 año'].append(item['value'][2])
  df_retornos_todos['100 año'].append(item['value'][3])


df_retornos_todos = pd.DataFrame(df_retornos_todos)

df_retornos_todos.to_csv(os.path.join(path_save_figures, 'df_retornos_CC_somero.csv'),
                         decimal=',',
                         sep=';',
                         index=True)

df_retornos_todos

## Retorno profundos

In [ ]:
magnitud_minima_retorno = 0.1
magnitud_maxima_retorno = 7
steps_magnitudes = 0.2
print_texto = False

### FC todos

In [ ]:
df_FC.describe().round(2)

In [ ]:
total_time = df_FC['Tiempo'].max() - df_FC['Tiempo'].min()
total_time = total_time / np.timedelta64(1, 'D')
total_time = total_time / 365.25

total_time # en años

In [ ]:
dict_retornos_FC_todos = sup.stats.periodo_retorno(a_value = df_GR_todos.loc['a_mean', 'FC profundo'],
                                                   b_value = df_GR_todos.loc['b_mean', 'FC profundo'],
                                                   magnitud_minima = magnitud_minima_retorno,
                                                   magnitud_maxima = magnitud_maxima_retorno,
                                                   steps_ = steps_magnitudes,
                                                   periodo_years = total_time,
                                                   retorno_years = [1, 10, 50, 100],
                                                   texto = print_texto)

dict_retornos_FC_todos.keys()

In [ ]:
dict_retornos_FC_todos.keys()

In [ ]:
df_retornos_todos = {'Magnitud': [],
                     'Periodo retorno': [],
                     '1 año': [],
                     '10 año': [],
                     '50 año': [],
                     '100 año': []}

for mag, retorno in zip(dict_retornos_FC_todos['m'],
                        dict_retornos_FC_todos['theta_years']):
  df_retornos_todos['Magnitud'].append(mag)
  df_retornos_todos['Periodo retorno'].append(retorno)

for item in dict_retornos_FC_todos['P_retornos']:
  df_retornos_todos['1 año'].append(item['value'][0])
  df_retornos_todos['10 año'].append(item['value'][1])
  df_retornos_todos['50 año'].append(item['value'][2])
  df_retornos_todos['100 año'].append(item['value'][3])


df_retornos_todos = pd.DataFrame(df_retornos_todos)

df_retornos_todos.to_csv(os.path.join(path_save_figures, 'df_retornos_FC_profundo.csv'),
                         decimal=',',
                         sep=';',
                         index=True)

df_retornos_todos

### BSN todos

In [ ]:
df_BSN.describe().round(2)

In [ ]:
total_time = df_BSN['Tiempo'].max() - df_BSN['Tiempo'].min()
total_time = total_time / np.timedelta64(1, 'D')
total_time = total_time / 365.25

total_time # en años

In [ ]:
dict_retornos_BSN_todos = sup.stats.periodo_retorno(a_value = df_GR_todos.loc['a_mean', 'BSN profundo'],
                                                   b_value = df_GR_todos.loc['b_mean', 'BSN profundo'],
                                                   magnitud_minima = magnitud_minima_retorno,
                                                   magnitud_maxima = magnitud_maxima_retorno,
                                                   steps_ = steps_magnitudes,
                                                   periodo_years = total_time,
                                                   retorno_years = [1, 10, 50, 100],
                                                   texto = print_texto)

dict_retornos_BSN_todos.keys()

In [ ]:
dict_retornos_BSN_todos.keys()

In [ ]:
df_retornos_todos = {'Magnitud': [],
                     'Periodo retorno': [],
                     '1 año': [],
                     '10 año': [],
                     '50 año': [],
                     '100 año': []}

for mag, retorno in zip(dict_retornos_BSN_todos['m'],
                        dict_retornos_BSN_todos['theta_years']):
  df_retornos_todos['Magnitud'].append(mag)
  df_retornos_todos['Periodo retorno'].append(retorno)

for item in dict_retornos_BSN_todos['P_retornos']:
  df_retornos_todos['1 año'].append(item['value'][0])
  df_retornos_todos['10 año'].append(item['value'][1])
  df_retornos_todos['50 año'].append(item['value'][2])
  df_retornos_todos['100 año'].append(item['value'][3])


df_retornos_todos = pd.DataFrame(df_retornos_todos)

df_retornos_todos.to_csv(os.path.join(path_save_figures, 'df_retornos_BSN_profundo.csv'),
                         decimal=',',
                         sep=';',
                         index=True)

df_retornos_todos

### CC todos

In [ ]:
df_CC.describe().round(2)

In [ ]:
total_time = df_CC['Tiempo'].max() - df_CC['Tiempo'].min()
total_time = total_time / np.timedelta64(1, 'D')
total_time = total_time / 365.25

total_time # en años

In [ ]:
dict_retornos_CC_todos = sup.stats.periodo_retorno(a_value = df_GR_todos.loc['a_mean', 'CC profundo'],
                                                   b_value = df_GR_todos.loc['b_mean', 'CC profundo'],
                                                   magnitud_minima = magnitud_minima_retorno,
                                                   magnitud_maxima = magnitud_maxima_retorno,
                                                   steps_ = steps_magnitudes,
                                                   periodo_years = total_time,
                                                   retorno_years = [1, 10, 50, 100],
                                                   texto = print_texto)

dict_retornos_CC_todos.keys()

In [ ]:
dict_retornos_CC_todos.keys()

In [ ]:
df_retornos_todos = {'Magnitud': [],
                     'Periodo retorno': [],
                     '1 año': [],
                     '10 año': [],
                     '50 año': [],
                     '100 año': []}

for mag, retorno in zip(dict_retornos_CC_todos['m'],
                        dict_retornos_CC_todos['theta_years']):
  df_retornos_todos['Magnitud'].append(mag)
  df_retornos_todos['Periodo retorno'].append(retorno)

for item in dict_retornos_CC_todos['P_retornos']:
  df_retornos_todos['1 año'].append(item['value'][0])
  df_retornos_todos['10 año'].append(item['value'][1])
  df_retornos_todos['50 año'].append(item['value'][2])
  df_retornos_todos['100 año'].append(item['value'][3])


df_retornos_todos = pd.DataFrame(df_retornos_todos)

df_retornos_todos.to_csv(os.path.join(path_save_figures, 'df_retornos_CC_profundo.csv'),
                         decimal=',',
                         sep=';',
                         index=True)

df_retornos_todos

# Dimension fractal

El cálculo de la correlación integral y de la dimensión fractal se realizará únicamente para los datos asociados al Campo Colorado y para los eventos con profundidades menores a la de interés, conforme a la definición adoptada en el Notebook 03.

In [ ]:
df_CC.head()

## Profundidad limite

In [ ]:
profunidad_corte = 60 # km Notebook 03

df_CC_somero = df_CC.copy()
df_CC_somero = df_CC_somero[df_CC_somero['Profundidad [km]'] <= profunidad_corte]
df_CC_somero.reset_index(drop=True, inplace=True)

df_CC_somero.info()

## Correlacion integral

In [ ]:
radios_ = np.logspace(0, 2, 50) # km
radios_

In [ ]:
cr = sup.stats.correlacion_integral_xy(x_km = df_CC_somero['X [km]'],
                                       y_km = df_CC_somero['Y [km]'],
                                       radio_en_km = radios_)

cr

## Dr

In [ ]:
slope, intercepto, otros = sup.stats.estimar_Dc(r_km = radios_,
                                                 C = cr,
                                                 idx_range=None)

slope, intercepto, otros

### Grafico

In [ ]:
datos_x = otros[0]
datos_y = otros[1]

xx = np.logspace(np.log10(radios_[0]), np.log10(radios_[-1]), 100)
yy = 10**(intercepto + slope*np.log10(xx))

fig, ax = plt.subplots(figsize=(6,4))

# curva C(r)
ax.loglog(radios_[otros[3]][~otros[2]],
          cr[otros[3]][~otros[2]],
          'o',
          ms=3,
          c='k',
          alpha=0.7)
cr_min = radios_[otros[3]][otros[2]][0]
cr_max = radios_[otros[3]][otros[2]][-1]
ax.loglog(radios_[otros[3]][otros[2]],
          cr[otros[3]][otros[2]],
          'o',
          ms=3,
          c='b',
          alpha=1.0,
          label=f'Range C(r) = [{cr_min:.2f}, {cr_max:.2f}]')

ax.loglog(xx,
          yy,
          '--',
          c='r',
          lw=1.5,
          label=f'Dc={slope:.2f}')

ax.set_xlabel('Distance r [km]')
ax.set_ylabel('Correlation Integral C(r)')
ax.grid(True, which='both', ls='--', c='gray', alpha=0.7)
ax.legend()

ax.set_title(f'CC Depth $\leq{profunidad_corte}$ km')
plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'fractal_CC_general',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

## Incremento en profundidad

Se procederá a calcular la variación de la dimensión fractal a medida que aumenta la profundidad. El área de análisis se mantendrá fija, dado que no se dispone de un número suficiente de eventos para definir ventanas espaciales (cajas) independientes de forma robusta.

In [ ]:
punto_xy_CampoColorado = (4919222.973, 2308553.476) # x, y en metros

df_CC_somero['X [m] centrado'] = df_CC_somero['X [m]'] - punto_xy_CampoColorado[0]
df_CC_somero['Y [m] centrado'] = df_CC_somero['Y [m]'] - punto_xy_CampoColorado[1]

df_CC_somero['X [km] centrado'] = df_CC_somero['X [m] centrado'] / 1000
df_CC_somero['Y [km] centrado'] = df_CC_somero['Y [m] centrado'] /1000

df_CC_somero.describe().round(2)

### Histograma profundidad

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(8,4))

sup.plots.histograma_plot(df= df_CC_somero,
                          columna= 'Profundidad [km]',
                          ax_h= ax[0],
                          dist_min= df_CC_somero['Profundidad [km]'].min(),
                          step_= 2)

ax[0].set_xlabel('Depth [km]')


sup.plots.histograma_plot(df= df_CC_somero,
                          columna= 'Error Profundidad [km]',
                          ax_h= ax[1],
                          dist_min= 0,
                          step_= 2)
ax[1].set_xlabel('Depth Error [km]')

plt.suptitle(f'CC Catalog - Depth $\leq{profunidad_corte}$ km')

plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'histograma_CC_profundidad',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

In [ ]:
steps_prof = 2
profundidades_ = np.arange(2, profunidad_corte, steps_prof)

profundidades_

In [ ]:
for prof_ in profundidades_:
  print(prof_, ' km --> ', (df_CC_somero['Profundidad [km]'] <= prof_).sum())

In [ ]:
for prof_ in profundidades_:
  filtro_prof = df_CC_somero['Profundidad [km]'] <= prof_
  filtro_distancia = df_CC_somero['Distancia [km]'] <= prof_
  print(prof_, ' km --> ', (filtro_prof & filtro_distancia).sum())

In [ ]:
lista_profundidades = []
lista_Dc = []


for prof_ in profundidades_:
  filtro_prof = df_CC_somero['Profundidad [km]'] <= prof_
  df_CC_someto_temp_ = df_CC_somero[filtro_prof].copy()
  df_CC_someto_temp_.reset_index(drop=True, inplace=True)

  cr = sup.stats.correlacion_integral_xy(x_km = df_CC_someto_temp_['X [km]'],
                                        y_km = df_CC_someto_temp_['Y [km]'],
                                        radio_en_km = radios_)

  slope, intercepto, otros = sup.stats.estimar_Dc(r_km = radios_,
                                                  C = cr,
                                                  idx_range=None)

  lista_profundidades.append(prof_)
  lista_Dc.append(slope)

  datos_x = otros[0]
  datos_y = otros[1]

  xx = np.logspace(np.log10(radios_[0]), np.log10(radios_[-1]), 100)
  yy = 10**(intercepto + slope*np.log10(xx))

  fig, ax = plt.subplots(figsize=(6,4))

  # curva C(r)
  ax.loglog(radios_[otros[3]][~otros[2]],
            cr[otros[3]][~otros[2]],
            'o',
            ms=3,
            c='k',
            alpha=0.7)
  cr_min = radios_[otros[3]][otros[2]][0]
  cr_max = radios_[otros[3]][otros[2]][-1]
  ax.loglog(radios_[otros[3]][otros[2]],
            cr[otros[3]][otros[2]],
            'o',
            ms=3,
            c='b',
            alpha=0.7,
            label=f'Range C(r) = [{cr_min:.2f}, {cr_max:.2f}]')

  ax.loglog(xx,
            yy,
            '--',
            c='r',
            lw=1.5,
            label=f'Dc={slope:.2f}')

  ax.set_xlabel('Distance r [km]')
  ax.set_ylabel('Correlation Integral C(r)')
  ax.grid(True, which='both', ls='--', alpha=0.5)
  ax.legend()

  ax.set_title(f'CC Depth $\leq{prof_}$ km')
  plt.tight_layout()

  sup.plots.save_plot(path_save_figuras = path_save_figures,
                      file_name = f'fractal_CC_prof_{prof_}_km',
                      formato = 'png',
                      dpi_ = 300)

  plt.show()
  print('\n')

### Grafico Profundidad-Dc

In [ ]:
lista_profundidades = np.array(lista_profundidades)
lista_Dc = np.array(lista_Dc)

fig, ax = plt.subplots(figsize=(6,4))

# curva C(r)
ax.scatter(lista_profundidades,
           lista_Dc,
           c='k')

ax.set_xlabel('Depth [km]')
ax.set_ylabel('Fractal dimension (Dc)')
ax.grid(True, which='both', ls='--', c='gray', alpha=0.7)
#ax.legend()

ax.set_title(f'CC Catalog')
plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'fractal_CC_Profundidad_vs_Dc',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

## Ventana de profundidad

In [ ]:
steps_prof = 1
ventana_profundidad = 5
profundidades_ = np.arange(0, profunidad_corte+steps_prof-ventana_profundidad, steps_prof)

profundidades_

In [ ]:
for prof_ in profundidades_:
  friltro_prof = (df_CC_somero['Profundidad [km]'] >= (prof_)) & (df_CC_somero['Profundidad [km]'] <= prof_ + ventana_profundidad)
  print(f'Profundidad {prof_}-{prof_+ventana_profundidad}= ', friltro_prof.sum(), ' eventos')

In [ ]:
lista_profundidades = []
lista_Dc = []


for prof_ in profundidades_:
  filtro_prof = (df_CC_somero['Profundidad [km]'] >= (prof_)) & (df_CC_somero['Profundidad [km]'] <= prof_ + ventana_profundidad)

  df_CC_someto_temp_ = df_CC_somero[filtro_prof].copy()
  df_CC_someto_temp_.reset_index(drop=True, inplace=True)

  cr = sup.stats.correlacion_integral_xy(x_km = df_CC_someto_temp_['X [km]'],
                                        y_km = df_CC_someto_temp_['Y [km]'],
                                        radio_en_km = radios_)

  slope, intercepto, otros = sup.stats.estimar_Dc(r_km = radios_,
                                                  C = cr,
                                                  idx_range=None)

  lista_profundidades.append(prof_ + (ventana_profundidad/2))
  lista_Dc.append(slope)

  datos_x = otros[0]
  datos_y = otros[1]

  xx = np.logspace(np.log10(radios_[0]), np.log10(radios_[-1]), 100)
  yy = 10**(intercepto + slope*np.log10(xx))

  fig, ax = plt.subplots(figsize=(6,4))

  # curva C(r)
  ax.loglog(radios_[otros[3]][~otros[2]],
            cr[otros[3]][~otros[2]],
            'o',
            ms=3,
            c='k',
            alpha=0.7)
  cr_min = radios_[otros[3]][otros[2]][0]
  cr_max = radios_[otros[3]][otros[2]][-1]
  ax.loglog(radios_[otros[3]][otros[2]],
            cr[otros[3]][otros[2]],
            'o',
            ms=3,
            c='b',
            alpha=0.7,
            label=f'Range C(r) = [{cr_min:.2f}, {cr_max:.2f}]')

  ax.loglog(xx,
            yy,
            '--',
            c='r',
            lw=1.5,
            label=f'Dc={slope:.2f}')

  ax.set_xlabel('Distance r [km]')
  ax.set_ylabel('Correlation Integral C(r)')
  ax.grid(True, which='both', ls='--', alpha=0.5)
  ax.legend()

  ax.set_title(f'CC Depth {prof_}-{prof_+ventana_profundidad} km')
  plt.tight_layout()

  sup.plots.save_plot(path_save_figuras = path_save_figures,
                      file_name = f'fractal_CC_RangoProf_{prof_}-{prof_+ventana_profundidad}_km',
                      formato = 'png',
                      dpi_ = 300)

  plt.show()
  print('\n')

### Grafico Profundidad-Dc

In [ ]:
lista_profundidades = np.array(lista_profundidades)
lista_Dc = np.array(lista_Dc)

fig, ax = plt.subplots(figsize=(6,4))

# curva C(r)
ax.errorbar(x = lista_profundidades,
            y = lista_Dc,
            xerr = ventana_profundidad/2,
            fmt = '--o',
            ms=5,
            c='k')

ax.set_xlabel(f'Central depth of window [km] - window width = {ventana_profundidad} km')
ax.set_ylabel('Fractal dimension (Dc)')
ax.grid(True, which='both', ls='--', c='gray', alpha=0.7)
ax.set_xticks(np.arange(0, profunidad_corte+ventana_profundidad, ventana_profundidad))
#ax.legend()

ax.set_title(f'CC Catalog')
plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'fractal_CC_Profundidad_vs_Dc_ventana',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

# Fin